# Quantización + fine-tuning MLP27K Montmeló para FINN v4 CLEAN

Este notebook parte del último flujo que funcionó y lo deja preparado para reconstruir desde cero una carpeta FINN limpia.

Objetivo de esta versión:

1. Cargar el checkpoint `mlp_27k.ckpt`.
2. Crear y evaluar el modelo cuantizado W4A8.
3. Hacer fine-tuning cuantizado.
4. Exportar a ONNX/QONNX para FINN.
5. Crear `model.onnx` con entrada `UINT8`.
6. Borrar la carpeta antigua mezclada con stubs/XPM.
7. Generar una carpeta limpia: `output_rebuild_mlp27k_w4a8_ft_clean`.
8. Preparar una simulación Verilator limpia con un único stub AXI Stream sencillo.

Antes de ejecutar, asegúrate de que en la carpeta del notebook está:

```text
mlp_27k.ckpt
```


## Versión clean / rebuild desde cero

Esta versión evita reutilizar la carpeta vieja `output_final_mlp27k_w4a8_ft`, porque en esa carpeta se mezclaron pruebas con:

- XPM reales de Vivado.
- XPM parcheadas.
- Stub viejo.
- Stub nuevo.
- Varios `all_verilog_srcs_verilator_*.txt`.

La salida limpia del build será:

```text
output_rebuild_mlp27k_w4a8_ft_clean
```

Y la simulación Verilator limpia se creará dentro de:

```text
output_rebuild_mlp27k_w4a8_ft_clean/stitched_ip/verilator_sim
```


## Resultados ya conocidos

La red cuantizada + fine-tuned ya se comprobó con las 500 primeras imágenes:

```text
MAE_v = 0.5783
MAE_w = 0.0282
```

Por eso este notebook usa una evaluación corta solo para comprobar que todo carga bien y después lanza FINN.


## 0. Imports

In [1]:
from pathlib import Path
import os
import csv
import math
import random
import shutil
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import brevitas.nn as qnn
import brevitas.onnx as bo
from brevitas.quant import Int8Bias

from qonnx.core.modelwrapper import ModelWrapper
from qonnx.core.onnx_exec import execute_onnx
from qonnx.core.datatype import DataType
from qonnx.transformation.infer_shapes import InferShapes
from qonnx.transformation.fold_constants import FoldConstants
from qonnx.transformation.infer_datatypes import InferDataTypes
from qonnx.transformation.general import (
    GiveUniqueNodeNames,
    GiveReadableTensorNames,
    RemoveStaticGraphInputs,
    RemoveUnusedTensors,
)

print("Torch:", torch.__version__)
print("Working dir:", Path.cwd())


Torch: 1.7.1
Working dir: /home/miguel_tfg/FINN-v0.9-Stable/notebooks/UAH_example


## 1. Rutas y dataset

In [2]:
CKPT_PATH = Path("mlp_27k.ckpt")

candidate_dirs = [
    Path("output_final_miguel/Dataset_JdeRobot/DATA_12_16/counter_montmelo_holonomic"),
    Path("/home/miguel_tfg/FINN-v0.9-Stable/notebooks/UAH_example/output_final_miguel/Dataset_JdeRobot/DATA_12_16/counter_montmelo_holonomic"),
    Path("/tmp/Orig/DATA_12_16/counter_montmelo_holonomic"),
    Path("/tmp/Orig/Orig/DATA_12_16/counter_montmelo_holonomic"),
]

DATASET_DIR = None
for d in candidate_dirs:
    if (d / "data.csv").exists():
        DATASET_DIR = d
        break

if not CKPT_PATH.exists():
    raise FileNotFoundError("No encuentro mlp_27k.ckpt en la carpeta del notebook")

if DATASET_DIR is None:
    raise FileNotFoundError("No encuentro el data.csv del dataset")

CSV_PATH = DATASET_DIR / "data.csv"
print("Checkpoint:", CKPT_PATH.resolve())
print("Dataset:", DATASET_DIR)
print("CSV:", CSV_PATH)

VMIN = -0.6
VMAX = 13.0
WMIN = -math.pi
WMAX = math.pi


def denorm_v_01(vnorm):
    return (VMAX - VMIN) * vnorm + VMIN


def denorm_w_01(wnorm):
    return (WMAX - WMIN) * wnorm + WMIN


def norm_v_01(v):
    return (v - VMIN) / (VMAX - VMIN)


def norm_w_01(w):
    return (w - WMIN) / (WMAX - WMIN)


def image_to_flat_01(img_path):
    img = Image.open(img_path).convert("RGB")
    img = img.resize((16, 12))
    arr = np.asarray(img).astype(np.float32)
    arr = np.transpose(arr, (2, 0, 1)).reshape(-1)  # RRR GGG BBB
    arr = arr / 255.0
    return torch.tensor(arr, dtype=torch.float32).unsqueeze(0)


def image_to_flat_uint8_np(img_path):
    img = Image.open(img_path).convert("RGB")
    img = img.resize((16, 12))
    arr = np.asarray(img).astype(np.float32)
    arr = np.transpose(arr, (2, 0, 1)).reshape(-1)  # RRR GGG BBB
    return arr.reshape(1, 576).astype(np.float32)


def load_rows(max_rows=None):
    rows = []
    with open(CSV_PATH) as f:
        reader = csv.DictReader(f)
        for row in reader:
            img_path = DATASET_DIR / row["image_name"]
            if not img_path.exists():
                continue
            rows.append({
                "image_name": row["image_name"],
                "img_path": img_path,
                "v": float(row["v"]),
                "w": float(row["w"]),
            })
            if max_rows is not None and len(rows) >= max_rows:
                break
    return rows

rows_eval = load_rows(80)  # muestra corta para que el notebook no se haga eterno
rows_all = load_rows(None)
print("Frames eval:", len(rows_eval))
print("Frames total:", len(rows_all))


Checkpoint: /home/miguel_tfg/FINN-v0.9-Stable/notebooks/UAH_example/mlp_27k.ckpt
Dataset: output_final_miguel/Dataset_JdeRobot/DATA_12_16/counter_montmelo_holonomic
CSV: output_final_miguel/Dataset_JdeRobot/DATA_12_16/counter_montmelo_holonomic/data.csv
Frames eval: 80
Frames total: 6539


## 2. Modelo float de referencia

In [3]:
class FloatMLP27K(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(576, 32)
        self.fx = nn.Linear(32, 256)
        self.predictor = nn.Linear(256, 2)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fx(x))
        x = self.predictor(x)
        return x


def extract_state_dict(obj):
    if isinstance(obj, dict):
        for key in ["state_dict", "model_state_dict", "net", "model"]:
            if key in obj and isinstance(obj[key], dict):
                return obj[key]
        return obj
    return obj.state_dict()


def clean_state_dict(sd):
    out = {}
    for k, v in sd.items():
        if not torch.is_tensor(v):
            continue
        nk = k
        changed = True
        while changed:
            changed = False
            for pref in ["model.", "base_model.", "module.", "net."]:
                if nk.startswith(pref):
                    nk = nk[len(pref):]
                    changed = True
        nk = nk.replace("predictor.0.", "predictor.")
        out[nk] = v
    return out


obj = torch.load(str(CKPT_PATH), map_location="cpu")
sd = clean_state_dict(extract_state_dict(obj))

print("Tensores encontrados en checkpoint:")
for k, v in sd.items():
    if torch.is_tensor(v):
        print(k, tuple(v.shape))

float_model = FloatMLP27K()
float_model.load_state_dict(sd, strict=True)
float_model.eval()
print("Modelo float cargado OK")


Tensores encontrados en checkpoint:
fc1.weight (32, 576)
fc1.bias (32,)
fx.weight (256, 32)
fx.bias (256,)
predictor.weight (2, 256)
predictor.bias (2,)
Modelo float cargado OK


## 3. Función de evaluación contra CSV

In [4]:
def eval_pytorch_model(model, rows, name="model", print_first=60):
    model.eval()
    err_v, err_w = [], []
    pred_v, pred_w = [], []

    with torch.no_grad():
        for i, r in enumerate(rows):
            x = image_to_flat_01(r["img_path"])
            y = model(x)
            if hasattr(y, "value"):
                y = y.value
            y = y.view(-1).detach().cpu().numpy()

            vnorm = float(y[0])
            wnorm = float(y[1])
            v_pred = denorm_v_01(vnorm)
            w_pred = denorm_w_01(wnorm)

            v_csv = r["v"]
            w_csv = r["w"]

            err_v.append(v_pred - v_csv)
            err_w.append(w_pred - w_csv)
            pred_v.append(v_pred)
            pred_w.append(w_pred)

            if i < print_first:
                raw_v_csv = round(norm_v_01(v_csv) * 255)
                raw_w_csv = round(norm_w_01(w_csv) * 255)
                raw_v_pred = round(vnorm * 255)
                raw_w_pred = round(wnorm * 255)
                print(
                    f"{r['image_name']} | CSV v={v_csv:.4f} w={w_csv:.4f} "
                    f"raw_csv={raw_v_csv},{raw_w_csv} | "
                    f"{name} v={v_pred:.4f} w={w_pred:.4f} raw={raw_v_pred},{raw_w_pred}"
                )

    err_v = np.array(err_v)
    err_w = np.array(err_w)
    pred_v = np.array(pred_v)
    pred_w = np.array(pred_w)

    res = {
        "name": name,
        "frames": len(rows),
        "mae_v": float(np.mean(np.abs(err_v))),
        "mae_w": float(np.mean(np.abs(err_w))),
        "std_v": float(np.std(pred_v)),
        "std_w": float(np.std(pred_w)),
        "v_min": float(np.min(pred_v)),
        "v_max": float(np.max(pred_v)),
        "w_min": float(np.min(pred_w)),
        "w_max": float(np.max(pred_w)),
    }

    print()
    print(name)
    print("Frames:", res["frames"])
    print("MAE_v =", res["mae_v"])
    print("MAE_w =", res["mae_w"])
    print("v pred min/max/std =", res["v_min"], res["v_max"], res["std_v"])
    print("w pred min/max/std =", res["w_min"], res["w_max"], res["std_w"])

    return res


## 4. Evaluar modelo float

In [5]:
float_res = eval_pytorch_model(float_model, rows_eval, name="FloatMLP27K", print_first=60)
float_res


1.png | CSV v=5.0000 w=-0.2254 raw_csv=105,118 | FloatMLP27K v=4.8432 w=-0.1681 raw=102,121
2.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | FloatMLP27K v=4.8432 w=-0.1680 raw=102,121
3.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | FloatMLP27K v=4.8432 w=-0.1680 raw=102,121
4.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | FloatMLP27K v=4.8432 w=-0.1680 raw=102,121
5.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | FloatMLP27K v=4.8432 w=-0.1680 raw=102,121
6.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | FloatMLP27K v=4.8432 w=-0.1680 raw=102,121
7.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | FloatMLP27K v=4.8432 w=-0.1680 raw=102,121
8.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | FloatMLP27K v=4.8432 w=-0.1680 raw=102,121
9.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | FloatMLP27K v=4.8432 w=-0.1680 raw=102,121
10.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | FloatMLP27K v=4.8432 w=-0.1680 raw=102,121
11.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | FloatMLP27K v=4.8432 w=-0.168

{'name': 'FloatMLP27K',
 'frames': 80,
 'mae_v': 0.9571316301822661,
 'mae_w': 0.02950021172782604,
 'std_v': 0.0010391317792737187,
 'std_w': 0.11291819640916634,
 'v_min': 4.839579367637635,
 'v_max': 4.844088459014893,
 'w_min': -0.16820402689858982,
 'w_max': 0.15616156615033372}

## 5. Modelo cuantizado elegido: pesos 4 bits, activaciones 8 bits

In [6]:
class QuantMLP27K_W4A8_NoQuantOut(nn.Module):
    def __init__(self):
        super().__init__()

        self.quant_in = qnn.QuantIdentity(
            bit_width=8,
            return_quant_tensor=True,
            min_val=0.0,
            max_val=1.0
        )

        self.fc1 = qnn.QuantLinear(
            576, 32,
            bias=True,
            weight_bit_width=4,
            bias_quant=Int8Bias,
            return_quant_tensor=True
        )
        self.relu1 = qnn.QuantReLU(
            bit_width=8,
            return_quant_tensor=True
        )

        self.fx = qnn.QuantLinear(
            32, 256,
            bias=True,
            weight_bit_width=4,
            bias_quant=Int8Bias,
            return_quant_tensor=True
        )
        self.relu2 = qnn.QuantReLU(
            bit_width=8,
            return_quant_tensor=True
        )

        self.predictor = qnn.QuantLinear(
            256, 2,
            bias=True,
            weight_bit_width=4,
            bias_quant=Int8Bias,
            return_quant_tensor=True
        )

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = self.quant_in(x)
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.fx(x)
        x = self.relu2(x)
        x = self.predictor(x)
        if hasattr(x, "value"):
            x = x.value
        return x


def load_float_weights_into_quant(qmodel, sd):
    qsd = qmodel.state_dict()
    loaded = []
    for k, v in sd.items():
        if k in qsd and tuple(qsd[k].shape) == tuple(v.shape):
            qsd[k] = v
            loaded.append(k)
    qmodel.load_state_dict(qsd, strict=False)
    return loaded


qmodel = QuantMLP27K_W4A8_NoQuantOut()
loaded = load_float_weights_into_quant(qmodel, sd)
qmodel.eval()
print("Pesos cargados en cuantizado:")
for k in loaded:
    print(" ", k)
print("Total:", len(loaded))


Pesos cargados en cuantizado:
  fc1.weight
  fc1.bias
  fx.weight
  fx.bias
  predictor.weight
  predictor.bias
Total: 6


## 6. Evaluar cuantizado antes de fine-tuning

In [7]:
q_before_res = eval_pytorch_model(qmodel, rows_eval, name="Q_W4A8_before_finetune", print_first=60)
q_before_res


1.png | CSV v=5.0000 w=-0.2254 raw_csv=105,118 | Q_W4A8_before_finetune v=5.0482 w=-0.3801 raw=106,112
2.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | Q_W4A8_before_finetune v=5.0482 w=-0.3801 raw=106,112
3.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | Q_W4A8_before_finetune v=5.0482 w=-0.3801 raw=106,112
4.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | Q_W4A8_before_finetune v=5.0482 w=-0.3801 raw=106,112
5.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | Q_W4A8_before_finetune v=5.0482 w=-0.3801 raw=106,112
6.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | Q_W4A8_before_finetune v=5.0482 w=-0.3801 raw=106,112
7.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | Q_W4A8_before_finetune v=5.0482 w=-0.3801 raw=106,112
8.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | Q_W4A8_before_finetune v=5.0482 w=-0.3801 raw=106,112
9.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | Q_W4A8_before_finetune v=5.0482 w=-0.3801 raw=106,112
10.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | Q_W4A8_before_finetune 

{'name': 'Q_W4A8_before_finetune',
 'frames': 80,
 'mae_v': 0.9485992747545241,
 'mae_w': 0.2799553934865354,
 'std_v': 0.13474438754425186,
 'std_w': 0.06677192172868188,
 'v_min': 4.632344770431518,
 'v_max': 5.066469144821167,
 'w_min': -0.38857238875397027,
 'w_max': -0.19856325189359847}

## 7. Dataset y DataLoaders para fine-tuning

In [8]:
class MontmeloDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows[idx]
        x = image_to_flat_01(r["img_path"]).view(-1)
        y = torch.tensor([norm_v_01(r["v"]), norm_w_01(r["w"])], dtype=torch.float32)
        return x, y


random.seed(0)
rows_train_val = rows_all.copy()
random.shuffle(rows_train_val)

split = int(0.8 * len(rows_train_val))
train_rows = rows_train_val[:split]
val_rows = rows_train_val[split:]

train_ds = MontmeloDataset(train_rows)
val_ds = MontmeloDataset(val_rows)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=0, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=0)

print("Train samples:", len(train_ds))
print("Val samples:", len(val_ds))


Train samples: 5231
Val samples: 1308


## 8. Fine-tuning cuantizado

In [11]:
device = torch.device("cpu")
qmodel = qmodel.to(device)

loss_fn = nn.L1Loss()
optimizer = torch.optim.Adam(qmodel.parameters(), lr=1e-4, weight_decay=1e-6)

N_EPOCHS = 50
best_val_loss = 1e9
best_state = None

for epoch in range(1, N_EPOCHS + 1):
    qmodel.train()
    train_losses = []

    for x, y in train_loader:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()
        pred = qmodel(x)
        if hasattr(pred, "value"):
            pred = pred.value

        loss = loss_fn(pred, y)
        loss.backward()
        optimizer.step()

        train_losses.append(float(loss.item()))

    qmodel.eval()
    val_losses = []
    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(device)
            y = y.to(device)
            pred = qmodel(x)
            if hasattr(pred, "value"):
                pred = pred.value
            loss = loss_fn(pred, y)
            val_losses.append(float(loss.item()))

    train_loss = float(np.mean(train_losses))
    val_loss = float(np.mean(val_losses))

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {k: v.detach().cpu().clone() for k, v in qmodel.state_dict().items()}

    print(f"Epoch {epoch:03d}/{N_EPOCHS} | train_loss={train_loss:.6f} | val_loss={val_loss:.6f}")

print("Mejor val_loss:", best_val_loss)
qmodel.load_state_dict(best_state)
qmodel.eval()


Epoch 001/50 | train_loss=0.027334 | val_loss=0.029726
Epoch 002/50 | train_loss=0.027318 | val_loss=0.033575
Epoch 003/50 | train_loss=0.027880 | val_loss=0.026752
Epoch 004/50 | train_loss=0.027680 | val_loss=0.029411
Epoch 005/50 | train_loss=0.028128 | val_loss=0.029804
Epoch 006/50 | train_loss=0.028029 | val_loss=0.026823
Epoch 007/50 | train_loss=0.027845 | val_loss=0.028989
Epoch 008/50 | train_loss=0.027683 | val_loss=0.027641
Epoch 009/50 | train_loss=0.026620 | val_loss=0.024814
Epoch 010/50 | train_loss=0.027338 | val_loss=0.026225
Epoch 011/50 | train_loss=0.027791 | val_loss=0.026040
Epoch 012/50 | train_loss=0.029433 | val_loss=0.030102
Epoch 013/50 | train_loss=0.028417 | val_loss=0.032302
Epoch 014/50 | train_loss=0.029770 | val_loss=0.028059
Epoch 015/50 | train_loss=0.028057 | val_loss=0.025234
Epoch 016/50 | train_loss=0.028695 | val_loss=0.027559
Epoch 017/50 | train_loss=0.027832 | val_loss=0.026594
Epoch 018/50 | train_loss=0.028273 | val_loss=0.030618
Epoch 019/

QuantMLP27K_W4A8_NoQuantOut(
  (quant_in): QuantIdentity(
    (input_quant): ActQuantProxyFromInjector(
      (_zero_hw_sentinel): StatelessBuffer()
    )
    (act_quant): ActQuantProxyFromInjector(
      (_zero_hw_sentinel): StatelessBuffer()
      (fused_activation_quant_proxy): FusedActivationQuantProxy(
        (activation_impl): Identity()
        (tensor_quant): RescalingIntQuant(
          (int_quant): IntQuant(
            (float_to_int_impl): RoundSte()
            (tensor_clamp_impl): TensorClamp()
            (delay_wrapper): DelayWrapper(
              (delay_impl): _NoDelay()
            )
          )
          (scaling_impl): ParameterFromRuntimeStatsScaling(
            (stats_input_view_shape_impl): OverTensorView()
            (stats): _Stats(
              (stats_impl): AbsPercentile()
            )
            (restrict_clamp_scaling): _RestrictClampValue(
              (clamp_min_ste): Identity()
              (restrict_value_impl): FloatRestrictValue()
            

## 9. Evaluar cuantizado después del fine-tuning

In [12]:
q_after_res = eval_pytorch_model(qmodel, rows_eval, name="Q_W4A8_finetuned", print_first=60)
q_after_res


1.png | CSV v=5.0000 w=-0.2254 raw_csv=105,118 | Q_W4A8_finetuned v=4.9709 w=-0.1618 raw=104,121
2.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | Q_W4A8_finetuned v=4.9709 w=-0.1618 raw=104,121
3.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | Q_W4A8_finetuned v=4.9709 w=-0.1618 raw=104,121
4.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | Q_W4A8_finetuned v=4.9709 w=-0.1618 raw=104,121
5.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | Q_W4A8_finetuned v=4.9709 w=-0.1618 raw=104,121
6.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | Q_W4A8_finetuned v=4.9709 w=-0.1618 raw=104,121
7.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | Q_W4A8_finetuned v=4.9709 w=-0.1618 raw=104,121
8.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | Q_W4A8_finetuned v=4.9709 w=-0.1618 raw=104,121
9.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | Q_W4A8_finetuned v=4.9709 w=-0.1618 raw=104,121
10.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | Q_W4A8_finetuned v=4.9709 w=-0.1618 raw=104,121
11.png | CSV v=5.0000 w=-0.21

{'name': 'Q_W4A8_finetuned',
 'frames': 80,
 'mae_v': 0.8161381492018698,
 'mae_w': 0.031577287132894684,
 'std_v': 0.010026747934714267,
 'std_w': 0.11301664823760353,
 'v_min': 4.97085816860199,
 'v_max': 5.0003101348876955,
 'w_min': -0.16175913544870602,
 'w_max': 0.17735828944465482}

## 10. Guardar modelo fine-tuned

In [13]:
FT_PTH = Path("q_mlp27k_w4a8_finetuned.pth")
torch.save(qmodel.state_dict(), FT_PTH)
print("Guardado:", FT_PTH)
!ls -lh q_mlp27k_w4a8_finetuned.pth


Guardado: q_mlp27k_w4a8_finetuned.pth
-rw-r--r-- 1 1004 1005 127K Jul  8 10:37 q_mlp27k_w4a8_finetuned.pth


## 11. Exportar a ONNX FINN

In [14]:
FT_ONNX = Path("mlp27k_w4a8_finetuned.onnx")

bo.export_finn_onnx(
    qmodel,
    input_shape=(1, 576),
    export_path=str(FT_ONNX)
)

print("Exportado:", FT_ONNX)
!ls -lh mlp27k_w4a8_finetuned.onnx


Exportado: mlp27k_w4a8_finetuned.onnx
-rw-r--r-- 1 1004 1005 970K Jul  8 10:37 mlp27k_w4a8_finetuned.onnx


## 12. Tidy-up del ONNX cuantizado

In [15]:
MODEL_FOR_FINN_01 = Path("model_for_finn_w4a8_finetuned_01.onnx")

model = ModelWrapper(str(FT_ONNX))
model = model.transform(InferShapes())
model = model.transform(FoldConstants())
model = model.transform(GiveUniqueNodeNames())
model = model.transform(GiveReadableTensorNames())
model = model.transform(InferDataTypes())
model = model.transform(RemoveStaticGraphInputs())
model = model.transform(RemoveUnusedTensors())
model.save(str(MODEL_FOR_FINN_01))

print("Guardado:", MODEL_FOR_FINN_01)
print("Input:", model.graph.input[0].name, model.get_tensor_shape(model.graph.input[0].name), model.get_tensor_datatype(model.graph.input[0].name))
print("Output:", model.graph.output[0].name, model.get_tensor_shape(model.graph.output[0].name))


Guardado: model_for_finn_w4a8_finetuned_01.onnx
Input: global_in [1, 576] FLOAT32
Output: global_out [1, 2]


/home/miguel_tfg/FINN-v0.9-Stable/deps/qonnx/src/qonnx/core/modelwrapper.py:93: UserWarning: Some old-style domain attributes were automatically converted to new-style,
                i.e. domain=finn to domain=qonnx.custom_op.<general|fpgadataflow|...>
  warnings.warn(


## 13. Evaluar ONNX exportado contra CSV

In [16]:
def eval_onnx_01(onnx_path, rows, name="ONNX", print_first=60):
    model = ModelWrapper(str(onnx_path))
    model = model.transform(InferShapes())
    input_name = model.graph.input[0].name
    output_name = model.graph.output[0].name

    err_v, err_w = [], []
    pred_v, pred_w = [], []

    for i, r in enumerate(rows):
        x = image_to_flat_01(r["img_path"]).numpy().astype(np.float32)
        y_dict = execute_onnx(model, {input_name: x})
        y = y_dict[output_name].reshape(-1)

        vnorm = float(y[0])
        wnorm = float(y[1])
        v_pred = denorm_v_01(vnorm)
        w_pred = denorm_w_01(wnorm)
        v_csv = r["v"]
        w_csv = r["w"]

        err_v.append(v_pred - v_csv)
        err_w.append(w_pred - w_csv)
        pred_v.append(v_pred)
        pred_w.append(w_pred)

        if i < print_first:
            print(
                f"{r['image_name']} | CSV v={v_csv:.4f} w={w_csv:.4f} | "
                f"{name} v={v_pred:.4f} w={w_pred:.4f}"
            )

    err_v = np.array(err_v)
    err_w = np.array(err_w)
    pred_v = np.array(pred_v)
    pred_w = np.array(pred_w)

    res = {
        "name": name,
        "frames": len(rows),
        "mae_v": float(np.mean(np.abs(err_v))),
        "mae_w": float(np.mean(np.abs(err_w))),
        "std_v": float(np.std(pred_v)),
        "std_w": float(np.std(pred_w)),
    }
    print(res)
    return res

onnx_res = eval_onnx_01(MODEL_FOR_FINN_01, rows_eval, name="ONNX_W4A8_finetuned", print_first=60)
onnx_res


/home/miguel_tfg/FINN-v0.9-Stable/deps/qonnx/src/qonnx/util/basic.py:300: UserWarning: The values of tensor MultiThreshold_0_out0 can't be represented with the set datatype annotation (INT8), they will be rounded to match the datatype annotation.
  warnings.warn(


1.png | CSV v=5.0000 w=-0.2254 | ONNX_W4A8_finetuned v=4.9709 w=-0.1618
2.png | CSV v=5.0000 w=-0.2131 | ONNX_W4A8_finetuned v=4.9709 w=-0.1618
3.png | CSV v=5.0000 w=-0.2131 | ONNX_W4A8_finetuned v=4.9709 w=-0.1618
4.png | CSV v=5.0000 w=-0.2131 | ONNX_W4A8_finetuned v=4.9709 w=-0.1618
5.png | CSV v=5.0000 w=-0.2131 | ONNX_W4A8_finetuned v=4.9709 w=-0.1618
6.png | CSV v=5.0000 w=-0.2131 | ONNX_W4A8_finetuned v=4.9709 w=-0.1618
7.png | CSV v=5.0000 w=-0.2131 | ONNX_W4A8_finetuned v=4.9709 w=-0.1618
8.png | CSV v=5.0000 w=-0.2131 | ONNX_W4A8_finetuned v=4.9709 w=-0.1618
9.png | CSV v=5.0000 w=-0.2131 | ONNX_W4A8_finetuned v=4.9709 w=-0.1618
10.png | CSV v=5.0000 w=-0.2131 | ONNX_W4A8_finetuned v=4.9709 w=-0.1618
11.png | CSV v=5.0000 w=-0.2131 | ONNX_W4A8_finetuned v=4.9799 w=-0.1576
12.png | CSV v=5.0000 w=-0.2131 | ONNX_W4A8_finetuned v=4.9709 w=-0.1618
13.png | CSV v=5.0000 w=-0.2131 | ONNX_W4A8_finetuned v=4.9709 w=-0.1618
14.png | CSV v=5.0000 w=-0.2131 | ONNX_W4A8_finetuned v=4.97

{'name': 'ONNX_W4A8_finetuned',
 'frames': 80,
 'mae_v': 0.8161373385787011,
 'mae_w': 0.031577387781658514,
 'std_v': 0.010026833221249322,
 'std_w': 0.11301660432912204}

## 14. Crear versión con preprocesado /255 para entrada UINT8

In [17]:
# Esta celda crea un ONNX que espera bytes 0-255 y divide internamente entre 255.
# Es útil para que coincida con el testbench C++ que envía bytes de imagen.

try:
    from finn.util.pytorch import ToTensor
    from qonnx.transformation.merge_onnx_models import MergeONNXModels

    MODEL_FOR_FINN_UINT8 = Path("model_for_finn_w4a8_finetuned_uint8.onnx")

    model = ModelWrapper(str(MODEL_FOR_FINN_01))
    model = model.transform(InferShapes())
    global_inp_name = model.graph.input[0].name
    ishape = model.get_tensor_shape(global_inp_name)
    print("Shape del modelo 0-1:", ishape)

    PREPROC_ONNX = Path("preproc_div255_w4a8.onnx")
    totensor_pyt = ToTensor()
    bo.export_finn_onnx(totensor_pyt, ishape, str(PREPROC_ONNX))

    pre_model = ModelWrapper(str(PREPROC_ONNX))
    merged = model.transform(MergeONNXModels(pre_model))
    global_inp_name = merged.graph.input[0].name
    merged.set_tensor_datatype(global_inp_name, DataType["UINT8"])

    merged = merged.transform(InferShapes())
    merged = merged.transform(FoldConstants())
    merged = merged.transform(GiveUniqueNodeNames())
    merged = merged.transform(GiveReadableTensorNames())
    merged = merged.transform(InferDataTypes())
    merged = merged.transform(RemoveStaticGraphInputs())
    merged = merged.transform(RemoveUnusedTensors())

    merged.save(str(MODEL_FOR_FINN_UINT8))
    shutil.copyfile(MODEL_FOR_FINN_UINT8, "model.onnx")

    print("Modelo UINT8 para FINN guardado en:", MODEL_FOR_FINN_UINT8)
    print("Copiado como model.onnx")
    print("Input:", merged.graph.input[0].name, merged.get_tensor_shape(merged.graph.input[0].name), merged.get_tensor_datatype(merged.graph.input[0].name))

except Exception as e:
    print("No se pudo crear la versión UINT8 con preprocesado.")
    print("Error:", repr(e))
    print("Copio el modelo 0-1 como model.onnx")
    shutil.copyfile(MODEL_FOR_FINN_01, "model.onnx")


Shape del modelo 0-1: [1, 576]
Modelo UINT8 para FINN guardado en: model_for_finn_w4a8_finetuned_uint8.onnx
Copiado como model.onnx
Input: global_in [1, 576] UINT8


/home/miguel_tfg/FINN-v0.9-Stable/deps/qonnx/src/qonnx/transformation/infer_data_layouts.py:124: UserWarning: Assuming 2D input is NC
  warnings.warn("Assuming 2D input is NC")


## 15. Evaluar model.onnx con entrada UINT8

In [18]:
# Si la celda anterior creó el preprocesado, model.onnx espera entrada 0-255.
# Si no, puede esperar entrada 0-1. Mira el datatype impreso en la celda anterior.

model = ModelWrapper("model.onnx")
model = model.transform(InferShapes())
input_name = model.graph.input[0].name
output_name = model.graph.output[0].name
print("Input:", input_name, model.get_tensor_shape(input_name), model.get_tensor_datatype(input_name))
print("Output:", output_name, model.get_tensor_shape(output_name))

err_v, err_w = [], []
for i, r in enumerate(rows_eval):
    # probamos con uint8 0-255 porque model.onnx debería incluir /255
    x = image_to_flat_uint8_np(r["img_path"])
    y_dict = execute_onnx(model, {input_name: x})
    y = y_dict[output_name].reshape(-1)
    v_pred = denorm_v_01(float(y[0]))
    w_pred = denorm_w_01(float(y[1]))
    err_v.append(v_pred - r["v"])
    err_w.append(w_pred - r["w"])
    if i < 500:
        print(f"{r['image_name']} | CSV v={r['v']:.4f} w={r['w']:.4f} | model.onnx v={v_pred:.4f} w={w_pred:.4f}")

err_v = np.array(err_v)
err_w = np.array(err_w)
print("MAE_v =", np.mean(np.abs(err_v)))
print("MAE_w =", np.mean(np.abs(err_w)))


Input: global_in [1, 576] UINT8
Output: global_out [1, 2]


/home/miguel_tfg/FINN-v0.9-Stable/deps/qonnx/src/qonnx/util/basic.py:300: UserWarning: The values of tensor MultiThreshold_0_out0 can't be represented with the set datatype annotation (INT8), they will be rounded to match the datatype annotation.
  warnings.warn(


1.png | CSV v=5.0000 w=-0.2254 | model.onnx v=4.9709 w=-0.1618
2.png | CSV v=5.0000 w=-0.2131 | model.onnx v=4.9709 w=-0.1618
3.png | CSV v=5.0000 w=-0.2131 | model.onnx v=4.9709 w=-0.1618
4.png | CSV v=5.0000 w=-0.2131 | model.onnx v=4.9709 w=-0.1618
5.png | CSV v=5.0000 w=-0.2131 | model.onnx v=4.9709 w=-0.1618
6.png | CSV v=5.0000 w=-0.2131 | model.onnx v=4.9709 w=-0.1618
7.png | CSV v=5.0000 w=-0.2131 | model.onnx v=4.9709 w=-0.1618
8.png | CSV v=5.0000 w=-0.2131 | model.onnx v=4.9709 w=-0.1618
9.png | CSV v=5.0000 w=-0.2131 | model.onnx v=4.9709 w=-0.1618
10.png | CSV v=5.0000 w=-0.2131 | model.onnx v=4.9709 w=-0.1618
11.png | CSV v=5.0000 w=-0.2131 | model.onnx v=4.9799 w=-0.1576
12.png | CSV v=5.0000 w=-0.2131 | model.onnx v=4.9709 w=-0.1618
13.png | CSV v=5.0000 w=-0.2131 | model.onnx v=4.9709 w=-0.1618
14.png | CSV v=5.0000 w=-0.2131 | model.onnx v=4.9799 w=-0.1576
15.png | CSV v=5.0000 w=-0.2131 | model.onnx v=4.9709 w=-0.1618
16.png | CSV v=5.0000 w=-0.2131 | model.onnx v=4.

## 16. FINN_BUILD_DIR

In [19]:
# Directorio temporal limpio para FINN
os.environ["FINN_BUILD_DIR"] = "/tmp/FINN_build_mlp27k_w4a8_ft_clean"
Path(os.environ["FINN_BUILD_DIR"]).mkdir(parents=True, exist_ok=True)
print("FINN_BUILD_DIR =", os.environ["FINN_BUILD_DIR"])


FINN_BUILD_DIR = /tmp/FINN_build_mlp27k_w4a8_ft_clean


## 17. Intentar build_dataflow

In [20]:
# ============================================================
# Variables de entorno Vivado necesarias para FINN
# ============================================================

VIVADO_PATH = Path("/tools/Xilinx/Vivado/2022.2")

if not VIVADO_PATH.exists():
    raise FileNotFoundError(f"No existe VIVADO_PATH: {VIVADO_PATH}")

os.environ["VIVADO_PATH"] = str(VIVADO_PATH)
os.environ["XILINX_VIVADO"] = str(VIVADO_PATH)
os.environ["PATH"] = str(VIVADO_PATH / "bin") + ":" + os.environ.get("PATH", "")

print("VIVADO_PATH:", os.environ["VIVADO_PATH"])
print("XILINX_VIVADO:", os.environ["XILINX_VIVADO"])

VIVADO_PATH: /tools/Xilinx/Vivado/2022.2
XILINX_VIVADO: /tools/Xilinx/Vivado/2022.2


In [21]:
from pathlib import Path
import os
import shutil

from finn.builder.build_dataflow import build_dataflow_cfg
import finn.builder.build_dataflow_config as build_cfg


# ============================================================
# Parche robusto para FINN v0.9 / distutils copy_tree
# ============================================================

import setuptools._distutils.file_util as su_file_util
import setuptools._distutils.dir_util as su_dir_util
import distutils.file_util as du_file_util
import distutils.dir_util as du_dir_util


def patch_copy_file_module(file_util_module):
    original_copy_file = file_util_module.copy_file

    def copy_file_mkdir(src, dst, *args, **kwargs):
        Path(dst).parent.mkdir(parents=True, exist_ok=True)
        return original_copy_file(src, dst, *args, **kwargs)

    file_util_module.copy_file = copy_file_mkdir
    return copy_file_mkdir


patched_su_copy = patch_copy_file_module(su_file_util)
patched_du_copy = patch_copy_file_module(du_file_util)

su_dir_util.file_util.copy_file = patched_su_copy
du_dir_util.file_util.copy_file = patched_du_copy

print("Parche aplicado: copy_file crea carpetas padre automáticamente")


# ============================================================
# Build FINN limpio desde model.onnx
# ============================================================

class MemModeCompat:
    def __init__(self, value):
        self.value = value

    def __repr__(self):
        return f"MemModeCompat(value={self.value!r})"


OLD_MIXED_OUTPUT_DIR = Path("output_final_mlp27k_w4a8_ft")

FINAL_OUTPUT_DIR = Path("output_rebuild_mlp27k_w4a8_ft_clean").resolve()
FINN_BUILD_DIR = Path("/tmp/FINN_build_mlp27k_w4a8_ft_clean")

MODEL_TO_BUILD = Path("model.onnx")

if not MODEL_TO_BUILD.exists():
    raise FileNotFoundError(
        "No existe model.onnx. Ejecuta antes las celdas de exportación ONNX/UINT8."
    )

# Variables Vivado
VIVADO_PATH = Path("/tools/Xilinx/Vivado/2022.2")

if not VIVADO_PATH.exists():
    raise FileNotFoundError(f"No existe VIVADO_PATH: {VIVADO_PATH}")

os.environ["VIVADO_PATH"] = str(VIVADO_PATH)
os.environ["XILINX_VIVADO"] = str(VIVADO_PATH)
os.environ["PATH"] = str(VIVADO_PATH / "bin") + ":" + os.environ.get("PATH", "")

print("VIVADO_PATH:", os.environ["VIVADO_PATH"])
print("XILINX_VIVADO:", os.environ["XILINX_VIVADO"])

# Borrar carpeta vieja contaminada
if OLD_MIXED_OUTPUT_DIR.exists():
    shutil.rmtree(OLD_MIXED_OUTPUT_DIR)
    print("Borrada carpeta vieja mezclada:", OLD_MIXED_OUTPUT_DIR)
else:
    print("No existe la carpeta vieja:", OLD_MIXED_OUTPUT_DIR)

# Borrar output limpio anterior
if FINAL_OUTPUT_DIR.exists():
    shutil.rmtree(FINAL_OUTPUT_DIR)
    print("Borrado output limpio anterior:", FINAL_OUTPUT_DIR)

# Borrar temporal FINN anterior
if FINN_BUILD_DIR.exists():
    shutil.rmtree(FINN_BUILD_DIR)
    print("Borrado FINN_BUILD_DIR anterior:", FINN_BUILD_DIR)

# Crear carpetas base
FINN_BUILD_DIR.mkdir(parents=True, exist_ok=True)
FINAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

os.environ["FINN_BUILD_DIR"] = str(FINN_BUILD_DIR)

print("Creada carpeta output:", FINAL_OUTPUT_DIR)

try:
    shell_none = build_cfg.ShellFlowType.NONE
except Exception:
    shell_none = None

cfg = build_cfg.DataflowBuildConfig(
    output_dir=str(FINAL_OUTPUT_DIR),
    folding_config_file=None,
    auto_fifo_depths=True,
    mvau_wwidth_max=36,
    stitched_ip_gen_dcp=True,
    split_large_fifos=True,
    synth_clk_period_ns=10.0,
    board=None,
    fpga_part="xc7z020clg400-1",
    default_mem_mode="const",
    shell_flow_type=shell_none,
    generate_outputs=[
        build_cfg.DataflowOutputType.ESTIMATE_REPORTS,
        build_cfg.DataflowOutputType.STITCHED_IP,
    ],
    save_intermediate_models=True,
)

if isinstance(cfg.default_mem_mode, str):
    print("Parcheando default_mem_mode:", cfg.default_mem_mode)
    cfg.default_mem_mode = MemModeCompat(cfg.default_mem_mode)

print("FINN_BUILD_DIR:", os.environ["FINN_BUILD_DIR"])
print("Modelo:", MODEL_TO_BUILD.resolve())
print("Output dir:", FINAL_OUTPUT_DIR)
print("default_mem_mode:", cfg.default_mem_mode, type(cfg.default_mem_mode))
print("generate_outputs:", cfg.generate_outputs)

print("Lanzando build_dataflow limpio...")
build_dataflow_cfg(str(MODEL_TO_BUILD), cfg)
print("FINN build terminado")

Parche aplicado: copy_file crea carpetas padre automáticamente
VIVADO_PATH: /tools/Xilinx/Vivado/2022.2
XILINX_VIVADO: /tools/Xilinx/Vivado/2022.2
No existe la carpeta vieja: output_final_mlp27k_w4a8_ft
Borrado output limpio anterior: /home/miguel_tfg/FINN-v0.9-Stable/notebooks/UAH_example/output_rebuild_mlp27k_w4a8_ft_clean
Borrado FINN_BUILD_DIR anterior: /tmp/FINN_build_mlp27k_w4a8_ft_clean
Creada carpeta output: /home/miguel_tfg/FINN-v0.9-Stable/notebooks/UAH_example/output_rebuild_mlp27k_w4a8_ft_clean
Parcheando default_mem_mode: const
FINN_BUILD_DIR: /tmp/FINN_build_mlp27k_w4a8_ft_clean
Modelo: /home/miguel_tfg/FINN-v0.9-Stable/notebooks/UAH_example/model.onnx
Output dir: /home/miguel_tfg/FINN-v0.9-Stable/notebooks/UAH_example/output_rebuild_mlp27k_w4a8_ft_clean
default_mem_mode: MemModeCompat(value='const') <class '__main__.MemModeCompat'>
generate_outputs: [<DataflowOutputType.ESTIMATE_REPORTS: 'estimate_reports'>, <DataflowOutputType.STITCHED_IP: 'stitched_ip'>]
Lanzando build

## 18. Debug si falla o parece parado

Si parece que el build se queda parado, mira el log desde otra terminal del host:

```bash
cd ~/FINN-v0.9-Stable/notebooks/UAH_example
tail -f output_rebuild_mlp27k_w4a8_ft_clean/build_dataflow.log
```

Y comprueba si crece el directorio temporal:

```bash
du -sh /tmp/FINN_build_mlp27k_w4a8_ft_clean
find /tmp/FINN_build_mlp27k_w4a8_ft_clean -type f | wc -l
```


In [22]:
!tail -120 output_rebuild_mlp27k_w4a8_ft_clean/build_dataflow.log 2>/dev/null || true


[2026-07-08 10:45:38,344] copying /tmp/FINN_build_mlp27k_w4a8_ft_clean/vivado_stitch_proj_8ba8a6h2/finn_vivado_stitch_proj.runs/finn_design_StreamingFIFO_0_1_0_synth_1/rundef.js -> /home/miguel_tfg/FINN-v0.9-Stable/notebooks/UAH_example/output_rebuild_mlp27k_w4a8_ft_clean/stitched_ip/finn_vivado_stitch_proj.runs/finn_design_StreamingFIFO_0_1_0_synth_1
[2026-07-08 10:45:38,344] copying /tmp/FINN_build_mlp27k_w4a8_ft_clean/vivado_stitch_proj_8ba8a6h2/finn_vivado_stitch_proj.runs/finn_design_StreamingFIFO_0_1_0_synth_1/.vivado.begin.rst -> /home/miguel_tfg/FINN-v0.9-Stable/notebooks/UAH_example/output_rebuild_mlp27k_w4a8_ft_clean/stitched_ip/finn_vivado_stitch_proj.runs/finn_design_StreamingFIFO_0_1_0_synth_1
[2026-07-08 10:45:38,344] copying /tmp/FINN_build_mlp27k_w4a8_ft_clean/vivado_stitch_proj_8ba8a6h2/finn_vivado_stitch_proj.runs/finn_design_StreamingFIFO_0_1_0_synth_1/htr.txt -> /home/miguel_tfg/FINN-v0.9-Stable/notebooks/UAH_example/output_rebuild_mlp27k_w4a8_ft_clean/stitched_ip/f

In [24]:
onnx_files = sorted(Path("/tmp/FINN_build_mlp27k_w4a8_ft_clean").rglob("*.onnx"))
print("ONNX intermedios:", len(onnx_files))
for i, f in enumerate(onnx_files):
    print(i, f)

if onnx_files:
    last_model = onnx_files[-1]
    print("Inspeccionando:", last_model)
    m = ModelWrapper(str(last_model))
    for node in m.graph.node:
        print(node.name, node.op_type)


ONNX intermedios: 0


## 19. Preparar Verilator limpio

Después de que FINN genere correctamente `output_rebuild_mlp27k_w4a8_ft_clean`, esta parte crea una simulación limpia.

Puntos importantes:

- No usa las XPM reales de Vivado.
- No copia stubs antiguos de la carpeta vieja.
- Escribe desde cero un único stub `xpm_fifo_axis` sencillo tipo registro/skid-buffer.
- Crea un único listado limpio: `all_verilog_srcs_verilator_clean.txt`.


In [25]:
from pathlib import Path

OUT = Path("output_rebuild_mlp27k_w4a8_ft_clean")
STITCHED = OUT / "stitched_ip"
SIM = STITCHED / "verilator_sim"
STUBS = STITCHED / "verilator_stubs"

if not STITCHED.exists():
    raise FileNotFoundError(
        f"No existe {STITCHED}. Ejecuta antes el build_dataflow y comprueba que generó STITCHED_IP."
    )

SIM.mkdir(parents=True, exist_ok=True)
STUBS.mkdir(parents=True, exist_ok=True)

stub_text = r"""module xpm_fifo_axis #(
    parameter integer CDC_DEST_SYNC_FF    = 2,
    parameter string  CLOCKING_MODE       = "common_clock",
    parameter string  ECC_MODE            = "no_ecc",
    parameter integer FIFO_DEPTH          = 16,
    parameter string  FIFO_MEMORY_TYPE    = "auto",
    parameter integer PACKET_FIFO         = 0,
    parameter integer PROG_EMPTY_THRESH   = 10,
    parameter integer PROG_FULL_THRESH    = 10,
    parameter integer RD_DATA_COUNT_WIDTH = 1,
    parameter integer RELATED_CLOCKS      = 0,
    parameter integer SIM_ASSERT_CHK      = 0,
    parameter integer TDATA_WIDTH         = 32,
    parameter integer TDEST_WIDTH         = 1,
    parameter integer TID_WIDTH           = 1,
    parameter integer TUSER_WIDTH         = 1,
    parameter string  USE_ADV_FEATURES    = "1000",
    parameter integer WR_DATA_COUNT_WIDTH = 1
)(
    input  wire                         s_aclk,
    input  wire                         m_aclk,
    input  wire                         s_aresetn,

    input  wire [TDATA_WIDTH-1:0]       s_axis_tdata,
    input  wire [TDATA_WIDTH/8-1:0]     s_axis_tkeep,
    input  wire [TDATA_WIDTH/8-1:0]     s_axis_tstrb,
    input  wire [TUSER_WIDTH-1:0]       s_axis_tuser,
    input  wire [TID_WIDTH-1:0]         s_axis_tid,
    input  wire [TDEST_WIDTH-1:0]       s_axis_tdest,
    input  wire                         s_axis_tvalid,
    output wire                         s_axis_tready,
    input  wire                         s_axis_tlast,

    output reg  [TDATA_WIDTH-1:0]       m_axis_tdata,
    output reg  [TDATA_WIDTH/8-1:0]     m_axis_tkeep,
    output reg  [TDATA_WIDTH/8-1:0]     m_axis_tstrb,
    output reg  [TUSER_WIDTH-1:0]       m_axis_tuser,
    output reg  [TID_WIDTH-1:0]         m_axis_tid,
    output reg  [TDEST_WIDTH-1:0]       m_axis_tdest,
    output reg                          m_axis_tvalid,
    input  wire                         m_axis_tready,
    output reg                          m_axis_tlast,

    output wire                         almost_empty_axis,
    output wire                         almost_full_axis,
    output wire                         prog_empty_axis,
    output wire                         prog_full_axis,
    output wire                         sbiterr_axis,
    output wire                         dbiterr_axis,
    output wire [RD_DATA_COUNT_WIDTH-1:0] rd_data_count_axis,
    output wire [WR_DATA_COUNT_WIDTH-1:0] wr_data_count_axis,

    input  wire                         injectsbiterr_axis,
    input  wire                         injectdbiterr_axis
);

    assign almost_empty_axis  = 1'b0;
    assign almost_full_axis   = 1'b0;
    assign prog_empty_axis    = 1'b0;
    assign prog_full_axis     = 1'b0;
    assign sbiterr_axis       = 1'b0;
    assign dbiterr_axis       = 1'b0;
    assign rd_data_count_axis = {RD_DATA_COUNT_WIDTH{1'b0}};
    assign wr_data_count_axis = {WR_DATA_COUNT_WIDTH{1'b0}};

    // Stub simple para Verilator.
    // Acepta un dato si no hay dato pendiente o si el bloque siguiente está listo.
    assign s_axis_tready = (!m_axis_tvalid) || m_axis_tready;

    always @(posedge s_aclk) begin
        if (!s_aresetn) begin
            m_axis_tdata  <= {TDATA_WIDTH{1'b0}};
            m_axis_tkeep  <= {(TDATA_WIDTH/8){1'b0}};
            m_axis_tstrb  <= {(TDATA_WIDTH/8){1'b0}};
            m_axis_tuser  <= {TUSER_WIDTH{1'b0}};
            m_axis_tid    <= {TID_WIDTH{1'b0}};
            m_axis_tdest  <= {TDEST_WIDTH{1'b0}};
            m_axis_tlast  <= 1'b0;
            m_axis_tvalid <= 1'b0;
        end else begin
            if (s_axis_tready) begin
                m_axis_tvalid <= s_axis_tvalid;
                if (s_axis_tvalid) begin
                    m_axis_tdata <= s_axis_tdata;
                    m_axis_tkeep <= s_axis_tkeep;
                    m_axis_tstrb <= s_axis_tstrb;
                    m_axis_tuser <= s_axis_tuser;
                    m_axis_tid   <= s_axis_tid;
                    m_axis_tdest <= s_axis_tdest;
                    m_axis_tlast <= s_axis_tlast;
                end
            end
        end
    end

endmodule
"""
(STUBS / "xpm_fifo_axis_stub.v").write_text(stub_text)

print("Creado stub limpio:", STUBS / "xpm_fifo_axis_stub.v")
print("Carpeta simulación:", SIM)


Creado stub limpio: output_rebuild_mlp27k_w4a8_ft_clean/stitched_ip/verilator_stubs/xpm_fifo_axis_stub.v
Carpeta simulación: output_rebuild_mlp27k_w4a8_ft_clean/stitched_ip/verilator_sim


## 20. Copiar o crear `main.cpp`

Esta celda intenta copiar el `main.cpp` desde la carpeta antigua si todavía existe. Si la carpeta antigua se ha borrado, debes pegar tu `main.cpp` final en:

```text
output_rebuild_mlp27k_w4a8_ft_clean/stitched_ip/verilator_sim/main.cpp
```

El `main.cpp` tiene que ser el que envía imágenes 16x12 RGB como 576 bytes y lee la salida AXI Stream.


In [26]:
from pathlib import Path
import shutil

SIM = Path("output_rebuild_mlp27k_w4a8_ft_clean/stitched_ip/verilator_sim")
main_dst = SIM / "main.cpp"

candidates = [
    Path("output_final_mlp27k_w4a8_ft_backup_stubs/stitched_ip/verilator_sim/main.cpp"),
    Path("output_final_mlp27k_w4a8_ft/stitched_ip/verilator_sim/main.cpp"),
    Path("output_final_miguel/stitched_ip/verilator_sim/main.cpp"),
]

if main_dst.exists():
    print("Ya existe:", main_dst)
else:
    copied = False
    for c in candidates:
        if c.exists():
            shutil.copy(c, main_dst)
            print("Copiado main.cpp desde:", c)
            copied = True
            break
    if not copied:
        print("AVISO: no he encontrado main.cpp para copiar.")
        print("Copia manualmente tu main.cpp final en:")
        print(main_dst)

print("Estado main.cpp:", main_dst.exists(), main_dst)


AVISO: no he encontrado main.cpp para copiar.
Copia manualmente tu main.cpp final en:
output_rebuild_mlp27k_w4a8_ft_clean/stitched_ip/verilator_sim/main.cpp
Estado main.cpp: False output_rebuild_mlp27k_w4a8_ft_clean/stitched_ip/verilator_sim/main.cpp


## 21. Crear listado limpio de fuentes para Verilator

Esta celda crea `all_verilog_srcs_verilator_clean.txt` desde `all_verilog_srcs.txt`.

Elimina:

- XPM reales: `xpm_fifo`, `xpm_memory`, `xpm_cdc`.
- Otros stubs antiguos.
- Duplicados del wrapper.

Y añade solo:

```text
../verilator_stubs/xpm_fifo_axis_stub.v
```


In [27]:
from pathlib import Path

STITCHED = Path("output_rebuild_mlp27k_w4a8_ft_clean/stitched_ip")
src = STITCHED / "all_verilog_srcs.txt"
dst = STITCHED / "all_verilog_srcs_verilator_clean.txt"

if not src.exists():
    raise FileNotFoundError(f"No existe {src}")

lines = src.read_text().splitlines()
out = []

# Añadimos SOLO el stub AXI FIFO limpio
out.append("../verilator_stubs/xpm_fifo_axis_stub.v")

for line in lines:
    l = line.strip()
    if not l:
        continue

    low = l.lower()

    # Quitar XPM reales: ya se comprobó que pueden compilar parcheadas,
    # pero bloquean tready en Verilator.
    if "xpm_fifo" in low or "xpm_memory" in low or "xpm_cdc" in low:
        continue

    # Evitar stubs previos o carpetas de pruebas
    if "verilator_stubs" in low or "verilator_xpm" in low:
        continue

    # Evitar duplicar el wrapper si ya se pasa ../finn_design.v por línea de comandos
    if "finn_design_wrapper.v" in low:
        continue

    # Desde verilator_sim, las rutas relativas necesitan ../
    if l.startswith("/"):
        out.append(l)
    else:
        out.append("../" + l)

dst.write_text("\n".join(out) + "\n")

print("Creado:", dst)
print("Líneas:", len(out))
print("\nPrimeras 30 líneas:")
print("\n".join(out[:30]))

print("\nComprobación XPM/stub:")
for l in out:
    if "xpm" in l.lower() or "stub" in l.lower():
        print(l)


Creado: output_rebuild_mlp27k_w4a8_ft_clean/stitched_ip/all_verilog_srcs_verilator_clean.txt
Líneas: 656

Primeras 30 líneas:
../verilator_stubs/xpm_fifo_axis_stub.v
/tmp/FINN_build_mlp27k_w4a8_ft_clean/vivado_stitch_proj_8ba8a6h2/finn_vivado_stitch_proj.gen/sources_1/bd/finn_design/ipshared/8713/hdl/axis_infrastructure_v1_1_0.vh
/tmp/FINN_build_mlp27k_w4a8_ft_clean/vivado_stitch_proj_8ba8a6h2/finn_vivado_stitch_proj.gen/sources_1/bd/finn_design/ipshared/8713/hdl/axis_infrastructure_v1_1_vl_rfs.v
/tmp/FINN_build_mlp27k_w4a8_ft_clean/vivado_stitch_proj_8ba8a6h2/finn_vivado_stitch_proj.gen/sources_1/bd/finn_design/ipshared/dbd8/hdl/axis_data_fifo_v2_0_vl_rfs.v
/tmp/FINN_build_mlp27k_w4a8_ft_clean/vivado_stitch_proj_8ba8a6h2/finn_vivado_stitch_proj.gen/sources_1/bd/finn_design/ip/finn_design_fifo_0/synth/finn_design_fifo_0.v
/tmp/FINN_build_mlp27k_w4a8_ft_clean/vivado_stitch_proj_8ba8a6h2/finn_vivado_stitch_proj.gen/sources_1/bd/finn_design/ipshared/6c7b/StreamingFIFO_0_1.v
/tmp/FINN_buil

## 22. Compilar Verilator limpio

Ejecuta esta celda cuando ya exista `main.cpp`.


In [28]:
%%bash
cd output_rebuild_mlp27k_w4a8_ft_clean/stitched_ip/verilator_sim
rm -rf obj_dir

verilator -Wall --cc ../finn_design.v \
-f ../all_verilog_srcs_verilator_clean.txt \
--top-module finn_design_wrapper \
--exe main.cpp \
--build \
-I../finn_vivado_stitch_proj.gen/sources_1/bd/finn_design/ipshared/8713/hdl \
-CFLAGS "$(pkg-config --cflags opencv4)" \
-LDFLAGS "$(pkg-config --libs opencv4)" \
-Wno-fatal \
-Wno-MODDUP \
-Wno-TIMESCALEMOD \
-Wno-PINCONNECTEMPTY \
-Wno-DECLFILENAME \
-Wno-PINMISSING \
-Wno-COMBDLY \
-Wno-UNUSED \
-Wno-WIDTH \
-Wno-STMTDLY \
-Wno-UNDRIVEN \
2>&1 | tee compile_clean_stub_log.txt


bash: line 4: pkg-config: command not found
bash: line 4: pkg-config: command not found


%Error: /tmp/FINN_build_mlp27k_w4a8_ft_clean/vivado_stitch_proj_8ba8a6h2/finn_vivado_stitch_proj.gen/sources_1/bd/finn_design/ipshared/6336/hdl/verilog/Thresholding_Batch_0_hls_deadlock_kernel_monitor_top.vh:5:1: syntax error, unexpected assign
    5 | assign kernel_monitor_reset = ~ap_rst_n;
      | ^~~~~~
%Error: /tmp/FINN_build_mlp27k_w4a8_ft_clean/vivado_stitch_proj_8ba8a6h2/finn_vivado_stitch_proj.gen/sources_1/bd/finn_design/ipshared/6336/hdl/verilog/Thresholding_Batch_0_hls_deadlock_kernel_monitor_top.vh:13:1: syntax error, unexpected assign
   13 | assign axis_block_sigs[0] = ~grp_Thresholding_Batch_fu_546.in0_V_TDATA_blk_n;
      | ^~~~~~
%Error: /tmp/FINN_build_mlp27k_w4a8_ft_clean/vivado_stitch_proj_8ba8a6h2/finn_vivado_stitch_proj.gen/sources_1/bd/finn_design/ipshared/96f5/hdl/verilog/MatrixVectorActivation_0_hls_deadlock_kernel_monitor_top.vh:5:1: syntax error, unexpected assign
    5 | assign kernel_monitor_reset = ~ap_rst_n;
      | ^~~~~~
%Error: /tmp/FINN_build_mlp27k_

## 23. Ejecutar simulación limpia

Esta celda usa el dataset `DATA_12_16/counter_montmelo_holonomic`.


In [ ]:
%%bash
cd output_rebuild_mlp27k_w4a8_ft_clean/stitched_ip/verilator_sim

./obj_dir/Vfinn_design_wrapper \
/home/miguel_tfg/TFG_simulacion_1/Dataset_JdeRobot/DATA_12_16/counter_montmelo_holonomic \
/home/miguel_tfg/TFG_simulacion_1/Dataset_JdeRobot/DATA_12_16/counter_montmelo_holonomic/data.csv \
2>&1 | tee salida_clean_stub.txt


## 24. Resumen de la simulación

Lo primero que hay que mirar es que aparezca:

```text
[INPUT] bytes aceptados = 576
```

Si aparece `bytes aceptados = 0`, entonces el problema ya no viene de la carpeta antigua, sino del wrapper/diseño generado o de la lógica de reset/tready.


In [ ]:
%%bash
cd output_rebuild_mlp27k_w4a8_ft_clean/stitched_ip/verilator_sim

grep -E "Frame|\[INPUT\]|\[OUT_BEAT|\[OUTPUT_TOTAL\]|\[DRAIN\]|ERROR|Timeout|raw0|raw1|raw2|ciclos" salida_clean_stub.txt | head -180
